# 半监督 GNN 空间转录组细胞类型注释（转录本级别，不依赖细胞分割）

**架构变更说明**：
- 旧版：Xenium cell-level counts（依赖 Xenium 细胞分割结果）
- 新版：Xenium transcript → 5μm spot（不依赖细胞分割，从原始转录本重建）

**创新点 vs TopACT**：
- TopACT：固定多尺度半径聚合 + SVM，各尺度独立，无端到端学习
- 本方法：特征kNN图 + 空间kNN图，GNN消息传递自动学习聚合权重

In [1]:
# ============================================================
# Cell 1: 环境配置（与原版相同，新增 bin_size 参数）
# ============================================================
import os, sys, warnings
import numpy as np
import pandas as pd
import torch
import rpy2.robjects as ro

warnings.filterwarnings('ignore')

PATHS = {
    'raw': {
        'transcripts': './data/raw/xenium/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/transcripts.parquet',
        'flex_cache':  './data/cache/flex_data_processed.rds',
    },
    'cache': {
        'graph':   './data/cache/graph_spot/',
        'exports': './data/cache/exports/',
    },
    'output': {
        'models':      './results/models/',
        'predictions': './results/predictions/',
        'plots':       './plots/',
    },
}
for d in [*PATHS['output'].values(), PATHS['cache']['graph'], PATHS['cache']['exports']]:
    os.makedirs(d, exist_ok=True)

PARAMS = {
    # ── 转录本 → spot ──────────────────────────────────────────
    'bin_size':        5,      # μm。5μm 兼顾分辨率与节点规模
    'qv_threshold':   20,      # Q-score 阈值（与 TopACT 论文一致）
    'min_transcripts': 1,      # spot 最小转录本数

    # ── 图构建 ────────────────────────────────────────────────
    'k_feat':     15,          # 特征空间 kNN（跨模态连边）
    'k_spatial':  10,          # 空间 kNN（spot 内部）
    'val_ratio':   0.2,

    # ── 模型架构 ──────────────────────────────────────────────
    'hidden_dim':  256,
    'proj_dim':    512,
    'dropout':     0.5,

    # ── 训练 ──────────────────────────────────────────────────
    'n_epochs':    500,
    'lr':          1e-3,
    'weight_decay':5e-4,
    'patience':    40,
    'lr_factor':   0.5,
    'lr_patience': 15,
    'min_lr':      1e-5,
    'max_grad_norm':1.0,
    'warmup_epochs':30,

    # ── 半监督损失权重 ────────────────────────────────────────
    'lambda_mmd':  0.1,
    'lambda_ent':  0.01,
    'lambda_pl':   0.3,
    'pl_threshold':0.90,
    'pl_update_freq':10,

    'device': 'cpu',
    'seed':   42,
}

from utils import set_seed
set_seed(PARAMS['seed'])

from gpu_utils import list_gpus, select_gpu
print('\n当前服务器 GPU 状态：')
list_gpus(show_processes=True)
GPU_ID = 'auto'
PARAMS['device'] = select_gpu(GPU_ID, min_free_gb=8.0)
print(f'\n✅ 训练将使用设备: {PARAMS["device"]}')


当前服务器 GPU 状态：

┌──────────────────────────────────────────────────────────────────────┐
│ ID  型号                               空闲     已用     总计    占用 状态
├──────────────────────────────────────────────────────────────────────┤
│ [0]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [1]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [2]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [3]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [4]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [5]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [6]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
│ [7]  NVIDIA GeForce RTX 3080       19.5GB    0.2GB   19.7GB     1%  空闲 ✓
└──────────────────────────────────────────────────────────────────────┘

  安装 pynvml+psutil 可查看占用进程
[auto] 选择 GPU 0（空闲 19.5 GB）

已选 GPU 0：NVIDIA GeForce RTX 3080
  空闲：19.5 

## Step 1: 数据加载

从 Flex 缓存加载 scRNA（有标签），从 Xenium 原始转录本文件加载转录本数据。
**不使用 Xenium 的 per-cell counts**，避免依赖细胞分割。

In [2]:
%load_ext rpy2.ipython

In [3]:
%%R -o flex_expr -o flex_labels -o cell_types -o xenium_panel_genes
# ============================================================
# Cell 2: 从 Flex 缓存加载 scRNA 参考数据
# 注意：不再导出 xenium_expr 和 xenium_coords（已弃用）
# ============================================================
library(Seurat)

cat('Loading scRNA reference...\n')
flex_data   <- readRDS('./data/cache/flex_data_processed.rds')
xenium_data <- readRDS('./data/cache/xenium_data_processed.rds')  # 只用来获取 panel 基因

# ── 获取 Xenium panel 基因列表（用于特征对齐）──────────────────
xenium_panel_genes <- rownames(xenium_data)
common_genes       <- intersect(rownames(flex_data), xenium_panel_genes)
cat('Xenium panel:', length(xenium_panel_genes),
    '| Common with scRNA:', length(common_genes), '\n')

# ── 只导出 scRNA 数据（raw counts，Python 做归一化）────────────
flex_counts <- as.matrix(GetAssayData(flex_data, layer='counts')[common_genes, ])
flex_expr   <- t(flex_counts)   # (n_cells, n_genes)

# 标签编码
flex_labels_raw <- flex_data$cell_type
unique_labels   <- unique(flex_labels_raw)
label_map       <- setNames(seq_along(unique_labels) - 1L, unique_labels)
flex_labels     <- as.integer(label_map[flex_labels_raw])
cell_types      <- unique_labels

cat('scRNA cells:', nrow(flex_expr), '\n')
cat('Cell types:', length(cell_types), '\n')
cat(cell_types, sep='  ')
cat('\n')

R[write to console]: Loading required package: SeuratObject

R[write to console]: Loading required package: sp

R[write to console]: 
Attaching package: ‘SeuratObject’


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, t





    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    Loading scRNA reference...
Xenium panel: 5101 | Common with scRNA: 4912 


R[write to console]: Loading required package: BPCells



This message is displayed once every 8 hours.
scRNA cells: 17050 
Cell types: 16 
Tumor Associated Fibroblasts  Endothelial Cells  Stromal Associated Fibroblasts  T & NK Cells  Malignant Cells Lining Cyst  Proliferative Tumor Cells  Tumor Cells  Pericytes  Granulosa Cells  Macrophages  Smooth Muscle Cells  VEGFA+ Tumor Cells  MT-High, Jun+/Fos+ Tumor Cells  Fallopian Tube Epithelium  Inflammatory Tumor Cells  Ciliated Epithelial Cells


In [4]:
# ============================================================
# Cell 3: 加载 Xenium 原始转录本 + 构建 spot 图
# ============================================================
import os, pickle
import numpy as np
import torch
from utils_spot import (
    load_xenium_transcripts,
    bin_transcripts_to_spots,
    unified_normalize_spot,
    build_spot_graph,
)

GRAPH_CACHE_DIR = PATHS['cache']['graph']
CACHE_FILE  = os.path.join(GRAPH_CACHE_DIR,
    f"graph_kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
    f"_bin{PARAMS['bin_size']}_val{PARAMS['val_ratio']}.pt")
SCALER_FILE = os.path.join(GRAPH_CACHE_DIR, 'fitted_scaler.pkl')
SPOT_COORD_FILE = os.path.join(GRAPH_CACHE_DIR, 'spot_coords.npy')

# 公共基因列表（R 导出）
gene_list = list(xenium_panel_genes)
print(f'共用基因数: {len(gene_list)}')

FORCE_REBUILD = False
CACHE_OK = all(os.path.exists(p) for p in [CACHE_FILE, SCALER_FILE, SPOT_COORD_FILE])

if CACHE_OK and not FORCE_REBUILD:
    print('[Cache HIT] 从磁盘加载图...')
    ckpt = torch.load(CACHE_FILE, weights_only=False)
    data         = ckpt['data']
    class_weights = ckpt['class_weights']
    split_info   = ckpt['split_info']
    with open(SCALER_FILE, 'rb') as f:
        fitted_scaler = pickle.load(f)
    spot_coords = np.load(SPOT_COORD_FILE)
    print(f'  {data.x.shape[0]:,} 节点 | {data.edge_index.shape[1]:,} 边')

else:
    print('[Cache MISS] 从转录本重建图...')

    # Step 1: 加载并过滤转录本
    df_tx = load_xenium_transcripts(
        PATHS['raw']['transcripts'],
        gene_list=gene_list,
        qv_threshold=PARAMS['qv_threshold'],
    )

    # Step 2: 聚合为 spot 表达矩阵
    spot_expr, spot_coords = bin_transcripts_to_spots(
        df_tx,
        gene_list=gene_list,
        bin_size=PARAMS['bin_size'],
        min_transcripts=PARAMS['min_transcripts'],
    )

    # Step 3: 统一归一化（scaler 只在 scRNA 上 fit）
    print('  归一化...')
    scrna_norm, spot_norm, fitted_scaler = unified_normalize_spot(
        flex_expr, spot_expr
    )

    # Step 4: 构建联合图
    data, class_weights, split_info = build_spot_graph(
        scrna_norm   = scrna_norm,
        spot_norm    = spot_norm,
        spot_coords  = spot_coords,
        scrna_labels = flex_labels,
        k_feat       = PARAMS['k_feat'],
        k_spatial    = PARAMS['k_spatial'],
        val_ratio    = PARAMS['val_ratio'],
    )

    # 保存缓存
    torch.save({'data': data, 'class_weights': class_weights,
                'split_info': split_info}, CACHE_FILE)
    with open(SCALER_FILE, 'wb') as f:
        pickle.dump(fitted_scaler, f)
    np.save(SPOT_COORD_FILE, spot_coords)
    print(f'  已缓存 → {CACHE_FILE}')

print(f'\nClass weights: {class_weights.numpy().round(3)}')
print(f'Train / Val  : {data.train_mask.sum()} / {data.val_mask.sum()}')
print(f'Spot 节点数  : {data.spot_mask.sum():,}')
print(f'特征维度     : {data.x.shape[1]}')
print('预处理完成 ✓')

共用基因数: 5101
[Cache MISS] 从转录本重建图...
加载转录本文件: ./data/raw/xenium/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs/transcripts.parquet
  原始转录本数: 147,706,750
  列名: ['transcript_id', 'cell_id', 'overlaps_nucleus', 'feature_name', 'x_location', 'y_location', 'z_location', 'qv', 'fov_name', 'nucleus_distance', 'codeword_index', 'codeword_category', 'is_gene']
  Q≥20 过滤后: 130,166,566
  基因对齐后: 120,425,292 条转录本，5090 个基因
  bin_size=5μm → 2,126,986 个有效 spot
  特征维度: 5101
  每 spot 平均转录本数: 56.6
  归一化...


ValueError: X has 5101 features, but StandardScaler is expecting 4912 features as input.

## Step 3-4: GNN 训练（与原版完全相同）

GNN 模型、训练循环、损失函数全部不变。
只需把 `data` 的来源从 cell-level 换成 spot-level，其余代码一行不动。

In [ ]:
# ============================================================
# Cell 4a: 训练 GCN（带错误保护 + 进度条）
# ============================================================
import torch
from models import GCN, run_experiment
from gpu_utils import safe_train

if "gnn_results" not in dir():
    gnn_results = {}

result_gcn = safe_train(
    run_experiment,
    GCN, "GCN", data, list(cell_types), PARAMS, class_weights,
    # ── safe_train 专属参数 ───────────────────────────────
    model_name = "GCN",
    device     = PARAMS["device"],
    params     = PARAMS,
    fallback   = None,   # OOM 时返回 None，不影响后续 Cell
)

if result_gcn is not None:
    gnn_results["GCN"] = result_gcn
    torch.save(
        result_gcn["model"].state_dict(),
        f"{PATHS['output']['models']}/GCN_best.pt"
    )
    print(f"\n  模型已保存  Best val F1: {result_gcn['best_val_f1']:.4f}")
else:
    print("\n  ⚠️  GCN 训练失败，已跳过。检查上方错误信息后可重新运行本 Cell。")


In [ ]:
# ============================================================
# Cell 4b: 训练 GraphSAGE（带错误保护 + 进度条）
# ============================================================
import torch
from models import GraphSAGE, run_experiment
from gpu_utils import safe_train

if "gnn_results" not in dir():
    gnn_results = {}

result_sage = safe_train(
    run_experiment,
    GraphSAGE, "GraphSAGE", data, list(cell_types), PARAMS, class_weights,
    model_name = "GraphSAGE",
    device     = PARAMS["device"],
    params     = PARAMS,
    fallback   = None,
)

if result_sage is not None:
    gnn_results["GraphSAGE"] = result_sage
    torch.save(
        result_sage["model"].state_dict(),
        f"{PATHS['output']['models']}/GraphSAGE_best.pt"
    )
    print(f"\n  模型已保存  Best val F1: {result_sage['best_val_f1']:.4f}")
else:
    print("\n  ⚠️  GraphSAGE 训练失败，已跳过。")


In [ ]:
# ============================================================
# Cell 4c: 训练 GAT（AMP 显存优化 + 错误保护 + 进度条）
# ============================================================
import torch
from models_amp import run_gat_amp
from gpu_utils import safe_train, list_gpus

# ── GAT 专属显存优化开关 ──────────────────────────────────────
# 如果 OOM，按顺序逐步开启：
PARAMS["use_amp"]   = True    # Level 1: 混合精度，显存减少 ~45%（推荐常开）
PARAMS["gat_heads"] = 4       # Level 2: 头数不够就改 2，再减 ~30%
PARAMS["use_ckpt"]  = False   # Level 3: 梯度检查点，速度慢 20% 但最省显存

if "gnn_results" not in dir():
    gnn_results = {}

print("训练前 GPU 状态：")
list_gpus()

result_gat = safe_train(
    run_gat_amp,
    data, list(cell_types), PARAMS, class_weights,
    model_name = "GAT",
    device     = PARAMS["device"],
    params     = PARAMS,
    fallback   = None,
)

if result_gat is not None:
    gnn_results["GAT"] = result_gat
    torch.save(
        result_gat["model"].state_dict(),
        f"{PATHS['output']['models']}/GAT_best.pt"
    )
    print(f"\n  模型已保存  Best val F1: {result_gat['best_val_f1']:.4f}")
else:
    print("\n  ⚠️  GAT 训练失败，已跳过。")
    print("  建议：")
    print("    1. 查看上方 OOM 报告中的显存诊断")
    print("    2. 修改 PARAMS[\"use_amp\"]=True 或 PARAMS[\"gat_heads\"]=2")
    print("    3. 或切换 Cell 1 里的 GPU_ID 到空闲更多的 GPU")


## Step 5: 评估（需修改：spot 预测 → 聚合到细胞级别 → 与 scRNA 验证集对比）

In [ ]:
# ============================================================
# Cell 6: 评估（修改版）
# scRNA 验证集评估逻辑不变；新增 spot 预测可视化
# ============================================================
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from eval import compute_metrics
from utils_spot import aggregate_spot_to_cell

val_idx  = split_info['val_idx']
val_true = flex_labels[val_idx]

val_preds = {}

# ── GNN 在 scRNA 验证集上的预测（逻辑不变）──────────────────────
for name, res in gnn_results.items():
    model  = res['model'].to('cpu')
    data_c = data.to('cpu')
    model.eval()
    with torch.no_grad():
        log_probs = model(data_c)
    # 验证集节点就是 scRNA 中的 val 节点，下标直接用 val_mask
    val_pred_idx = log_probs[data_c.val_mask].argmax(dim=1).numpy()
    val_preds[name] = val_pred_idx

# ── 打印评估结果（与原版相同）───────────────────────────────────
print('=' * 65)
print('  Method Comparison on scRNA Val Set (Real Ground Truth)')
print('=' * 65)
print(f'{"Method":<12} {"Acc":>7} {"F1-Mac":>8} {"F1-Wei":>8} {"Kappa":>8}')
print('-' * 65)
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=len(cell_types))
    print(f'{method:<12} {m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} '
          f'{m["f1_weighted"]:>8.4f} {m["kappa"]:>8.4f}')
print('=' * 65)

# ── 新增：输出 spot 级别预测（用于空间可视化）──────────────────
best_model_name = max(gnn_results,
    key=lambda n: gnn_results[n]['best_val_f1'])
best_model = gnn_results[best_model_name]['model'].to('cpu').eval()

with torch.no_grad():
    log_probs_all = best_model(data.to('cpu'))
    proba_all = torch.softmax(log_probs_all, dim=1).numpy()

# Xenium spot 的预测概率
n_scrna = split_info['n_scrna']
spot_proba = proba_all[n_scrna:]
spot_pred_idx = spot_proba.argmax(axis=1)

# 保存 spot 级别预测（含坐标，用于空间可视化）
import pandas as pd
spot_df = pd.DataFrame({
    'x':         spot_coords[:, 0],
    'y':         spot_coords[:, 1],
    'pred_idx':  spot_pred_idx,
    'pred_label':[cell_types[i] for i in spot_pred_idx],
    'confidence': spot_proba.max(axis=1),
})
spot_df.to_csv(f'{PATHS["output"]["predictions"]}/spot_predictions_{best_model_name}.csv',
               index=False)
print(f'\nSpot 预测已保存（{len(spot_df):,} spots）')
print(f'使用模型: {best_model_name}，val F1: {gnn_results[best_model_name]["best_val_f1"]:.4f}')

In [ ]:
# ============================================================
# Cell 7: 空间可视化（新增，转录本级别特有）
# 展示 GNN 在空间上的预测结果，无需细胞分割
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(12, 10))

# 每个细胞类型一种颜色
n_types = len(cell_types)
cmap    = plt.cm.get_cmap('tab20', n_types)
colors  = {i: cmap(i) for i in range(n_types)}

# 散点图：每个 spot 按预测类型上色
for cls_idx, cls_name in enumerate(cell_types):
    mask = spot_df['pred_idx'] == cls_idx
    if mask.sum() == 0:
        continue
    ax.scatter(
        spot_df.loc[mask, 'x'],
        spot_df.loc[mask, 'y'],
        c=[colors[cls_idx]],
        s=0.5,
        alpha=0.6,
        label=cls_name,
        rasterized=True,
    )

ax.set_aspect('equal')
ax.set_title(f'GNN Spot-level Predictions ({best_model_name})\n'
             f'bin_size={PARAMS["bin_size"]}μm, No Cell Segmentation Required',
             fontsize=13)
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.legend(markerscale=8, bbox_to_anchor=(1.02, 1), loc='upper left',
          fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.savefig(f'{PATHS["output"]["plots"]}/spot_spatial_map_{best_model_name}.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('空间可视化已保存')